# Bitget `market` -- gap check (Classic)

Bitget's production SDK (`tribulnation.bitget`) has no `Market`/`PerpMarket`/`Exchange`
implementation -- only `earn`/`wallet`/`reporting`. `typed_bitget` does expose real spot
and futures (`mix`) trading surfaces though (order book, symbol/contract rules, place/
cancel order, open orders, fills, positions, funding), confirmed by independently reading
`classic/spot/{orderbook,symbols,order,account}` and `classic/mix/{market,order,position,
account}`. This notebook hand-maps that surface onto `Market` (spot) and `PerpMarket`
(mix USDT-FUTURES), the same way `earn`/`wallet`/`reporting` do above.

Split by account mode for the same reason as the other three pillars: Classic keeps spot
and futures balances in genuinely separate compartments (`spot.account.assets` vs.
`mix.account.get`, each with its own `available`/margin semantics), so a Classic-mode
integration's `position()`/`collateral()`/`available_notional()` are inherently
mode-specific -- not a case where market data happens to be identical across modes.

In [1]:
import asyncio
import os
from decimal import Decimal
from datetime import datetime, timedelta, timezone

from typing_extensions import Literal

from typed_bitget import Bitget
from typed_bitget.classic.spot.order.place import SpotLimitOrderRequest, SpotMarketOrderRequest
from typed_bitget.classic.mix.order.place import MixLimitOrderRequest, MixMarketOrderRequest
from dotenv import load_dotenv

from tribulnation.sdk.market import (
  Book,
  Collateral,
  FundingPayment,
  FundingRate,
  NextFunding,
  Order,
  OrderResponse,
  OrderState,
  PerpCollateral,
  PerpPosition,
  Position,
  Rules,
  Settings,
  Trade,
)

load_dotenv()

client = await Bitget.new(
  access_key=os.environ['BITGET_CLASSIC_ACCESS_KEY'],
  secret_key=os.environ['BITGET_CLASSIC_SECRET_KEY'],
  passphrase=os.environ['BITGET_CLASSIC_PASSPHRASE'],
).__aenter__()

SPOT_MARKETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
PERP_MARKETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
PRODUCT_TYPE = 'USDT-FUTURES'

## `Market` (spot)

In [2]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.classic.spot.orderbook(symbol=symbol, limit=levels)
  return Book(
    bids=[Book.Entry(price, qty) for price, qty in raw['bids']],
    asks=[Book.Entry(price, qty) for price, qty in raw['asks']],
  )

{symbol: await depth(symbol, levels=5) for symbol in SPOT_MARKETS}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('79666.88'), qty=Decimal('1.4384390000000000')), Book.Entry(price=Decimal('79666.81'), qty=Decimal('0.0811710000000000')), Book.Entry(price=Decimal('79666.8'), qty=Decimal('0.1957880000000000')), Book.Entry(price=Decimal('79666.67'), qty=Decimal('0.0033050000000000')), Book.Entry(price=Decimal('79666.22'), qty=Decimal('0.0005690000000000'))], asks=[Book.Entry(price=Decimal('79666.89'), qty=Decimal('0.3361760000000000')), Book.Entry(price=Decimal('79667.11'), qty=Decimal('0.0003950000000000')), Book.Entry(price=Decimal('79667.65'), qty=Decimal('0.0053880000000000')), Book.Entry(price=Decimal('79669.99'), qty=Decimal('0.0125510000000000')), Book.Entry(price=Decimal('79670.06'), qty=Decimal('0.1901900000000000'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2455.23'), qty=Decimal('12.4178000000000000')), Book.Entry(price=Decimal('2455.22'), qty=Decimal('0.0200000000000000')), Book.Entry(price=Decimal('2455.21'), qty=Decimal('0.0457000

In [3]:
async def rules(symbol: str, *, refetch: bool = False) -> Rules:
  raw = await client.classic.spot.symbols(symbol=symbol)
  sym = raw[0]
  return Rules(
    base=sym['baseCoin'],
    quote=sym['quoteCoin'],
    fee_asset=sym['quoteCoin'],
    tick_size=Decimal(10) ** -int(sym['pricePrecision']),
    step_size=Decimal(10) ** -int(sym['quantityPrecision']),
    fixed_min_qty=sym['minTradeAmount'] or None,
    min_value=sym['minTradeUSDT'],
    max_qty=sym['maxTradeAmount'],
    maker_fee=sym['makerFeeRate'],
    taker_fee=sym['takerFeeRate'],
    api=sym['status'] == 'online',
    details=sym,
  )

{symbol: await rules(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.01'), step_size=Decimal('0.000001'), fixed_min_qty=None, min_value=Decimal('1'), max_qty=Decimal('900000000000000000000'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.002'), taker_fee=Decimal('0.002'), api=True, details={'symbol': 'BTCUSDT', 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'minTradeAmount': Decimal('0'), 'maxTradeAmount': Decimal('900000000000000000000'), 'takerFeeRate': Decimal('0.002'), 'makerFeeRate': Decimal('0.002'), 'pricePrecision': 2, 'quantityPrecision': 6, 'quotePrecision': 8, 'minTradeUSDT': Decimal('1'), 'status': 'online', 'buyLimitPriceRatio': Decimal('0.02'), 'sellLimitPriceRatio': Decimal('0.02'), 'orderQuantity': 200, 'areaSymbol': 'no', 'maxLimitOrderValue': Decimal('20000000'), 'maxMarketOrderValue': Decimal('1000000'), 'openTime': datetime.datetime(2018, 7, 24, 17, 46, tzinfo=datetime.timezone.utc)}),
 'ETHUSDT': Rul

In [4]:
async def open_orders(symbol: str) -> list[OrderState]:
  raw = await client.classic.spot.order.open(symbol=symbol)
  out: list[OrderState] = []
  for o in raw:
    size = Decimal(o['size'])
    filled = Decimal(o['baseVolume'])
    sign = 1 if o['side'] == 'buy' else -1
    out.append(OrderState(
      id=o['orderId'],
      price=Decimal(o['basePrice']),
      qty=sign * size,
      filled_qty=sign * filled,
      active=True,  # this endpoint only lists currently-open orders
      details=o,
    ))
  return out

{symbol: await open_orders(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [2]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.classic.spot.order.fills(symbol=symbol, start_time=start, end_time=end)
  out: list[Trade] = []
  for f in raw:
    size = Decimal(f['size'])
    fee_detail = f['feeDetail']
    fee_amount = abs(Decimal(fee_detail['totalFee']))
    out.append(Trade(
      id=f['tradeId'],
      price=Decimal(f['priceAvg']),
      qty=size if f['side'] == 'buy' else -size,
      time=f['cTime'],
      maker=f['tradeScope'] == 'maker',
      fee=Trade.Fee(amount=fee_amount, asset=fee_detail['feeCoin']) if fee_amount else None,
      details=f,
    ))
  return out

end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await trades_history(symbol, start, end) for symbol in SPOT_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [ ]:
async def spot_asset(coin: str):
  raw = await client.classic.spot.account.assets(coin=coin)
  return raw[0] if raw else None


async def position(symbol: str) -> Position:
  base = symbol.removesuffix('USDT')  # SPOT_MARKETS are all *USDT pairs here
  balance = await spot_asset(base)
  size = (
    Decimal(balance['available']) + Decimal(balance['frozen']) + Decimal(balance['locked'])
    if balance else Decimal(0)
  )
  return Position(size=size)

{symbol: await position(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': Position(size=Decimal('2.665960000E-7')),
 'ETHUSDT': Position(size=Decimal('0.0013506509000000')),
 'SOLUSDT': Position(size=Decimal('0E-16'))}

In [7]:
async def available_notional(symbol: str) -> Decimal:
  balance = await spot_asset('USDT')
  return Decimal(balance['available']) if balance else Decimal(0)

{symbol: await available_notional(symbol) for symbol in SPOT_MARKETS}

{'BTCUSDT': Decimal('0.0030569541190000'),
 'ETHUSDT': Decimal('0.0030569541190000'),
 'SOLUSDT': Decimal('0.0030569541190000')}

In [ ]:
async def place_order(symbol: str, order: Order, *, settings: Settings = {}) -> OrderResponse:
  qty = Decimal(str(order['qty']))
  side = 'buy' if qty > 0 else 'sell'
  size = abs(qty)
  price = Decimal(str(order['price']))
  body: SpotLimitOrderRequest | SpotMarketOrderRequest
  if order['type'] == 'MARKET':
    body = {'symbol': symbol, 'side': side, 'orderType': 'market', 'force': 'gtc', 'size': size}
  elif order['type'] == 'POST_ONLY':
    body = {'symbol': symbol, 'side': side, 'orderType': 'limit', 'force': 'post_only', 'price': price, 'size': size}
  else:
    body = {'symbol': symbol, 'side': side, 'orderType': 'limit', 'force': 'gtc', 'price': price, 'size': size}
  raw = await client.classic.spot.order.place(body)
  return OrderResponse(id=raw['orderId'], details=raw)

# Not executed here -- would place a real order on the account.
await place_order('BTCUSDT', {'qty': Decimal('0.0001'), 'price': Decimal('20000'), 'type': 'LIMIT'})

In [ ]:
async def cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.classic.spot.order.cancel(symbol=symbol, order_id=id)

# Not executed here -- would cancel a real order on the account.
await cancel_order('BTCUSDT', '123456')

### Coverage assessment: `Market` (spot)

**Fully supported.** `classic.spot.{orderbook,symbols,order,account}` cover every
abstract method live-tested above with real market data. The account holds only dust
(BTC, ETH, USDT, USDC, AVAX, all well under a dollar) and has no open spot orders or
fills in the last 24 hours, so `open_orders`/`trades_history` come back empty and
`position()` returns those dust balances. `Rules`'s `tick_size`/`step_size` are derived
from `pricePrecision`/`quantityPrecision` (decimal place *counts*, not tick sizes
directly) rather than read off a dedicated field, since Bitget doesn't expose one.
`collateral()` is left unimplemented (matches the abstract base class's own default
`NotImplementedError` -- there's no separate "collateral bucket" concept for a spot
balance beyond the balance itself, same choice Binance's `poc/market.ipynb` makes for
its spot section).

`spot.account.assets` validates for a coin the account holds nothing in: SOL sends
`limitAvailable: null`, which the declaration (`Decimal | None`) admits.

## `PerpMarket` (Classic Mix, USDT-FUTURES)

In [4]:
# `classic.mix.market.orderbook`'s `limit` is a fixed enum (`'1' | '5' | '15' | '50' | 'max'`),
# unlike `classic.spot.orderbook`'s arbitrary `int` -- `levels` is restricted to the four
# numeric members and mapped here.
_DEPTH_LEVELS: dict[int, Literal['1', '5', '15', '50']] = {1: '1', 5: '5', 15: '15', 50: '50'}


async def perp_depth(symbol: str, *, levels: Literal[1, 5, 15, 50] | None = None) -> Book:
  raw = await client.classic.mix.market.orderbook(
    symbol=symbol, product_type=PRODUCT_TYPE,
    limit=_DEPTH_LEVELS[levels] if levels is not None else None,
  )
  return Book(
    bids=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q in raw['bids']],
    asks=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q in raw['asks']],
  )

{symbol: await perp_depth(symbol, levels=5) for symbol in PERP_MARKETS}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('79376.2'), qty=Decimal('1.9258')), Book.Entry(price=Decimal('79376.1'), qty=Decimal('1.4648')), Book.Entry(price=Decimal('79375.3'), qty=Decimal('0.0961')), Book.Entry(price=Decimal('79374.5'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('79373.8'), qty=Decimal('0.0001'))], asks=[Book.Entry(price=Decimal('79376.3'), qty=Decimal('2.57')), Book.Entry(price=Decimal('79376.4'), qty=Decimal('0.0013')), Book.Entry(price=Decimal('79376.8'), qty=Decimal('0.0005')), Book.Entry(price=Decimal('79376.9'), qty=Decimal('0.0001')), Book.Entry(price=Decimal('79377.4'), qty=Decimal('0.0013'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2451.0'), qty=Decimal('14.16')), Book.Entry(price=Decimal('2450.91'), qty=Decimal('0.09')), Book.Entry(price=Decimal('2450.9'), qty=Decimal('0.3')), Book.Entry(price=Decimal('2450.86'), qty=Decimal('0.1')), Book.Entry(price=Decimal('2450.82'), qty=Decimal('7.89'))], asks=[Book.Entry(price=Decimal('2451.01'), q

In [3]:
async def perp_rules(symbol: str, *, refetch: bool = False) -> Rules:
  raw = await client.classic.mix.market.contracts(product_type=PRODUCT_TYPE, symbol=symbol)
  c = raw[0]
  return Rules(
    base=c['baseCoin'],
    quote=c['quoteCoin'],
    fee_asset=c['quoteCoin'],
    tick_size=Decimal(10) ** -int(c['pricePlace']),
    step_size=c['sizeMultiplier'],
    fixed_min_qty=c['minTradeNum'],
    min_value=c['minTradeUSDT'],
    max_qty=Decimal(c['maxOrderQty']),
    maker_fee=c['makerFeeRate'],
    taker_fee=c['takerFeeRate'],
    api=c['symbolStatus'] == 'normal',
    details=c,
  )

{symbol: await perp_rules(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.1'), step_size=Decimal('0.0001'), fixed_min_qty=Decimal('0.0001'), min_value=Decimal('5'), max_qty=Decimal('1200'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.0002'), taker_fee=Decimal('0.0006'), api=True, details={'symbol': 'BTCUSDT', 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'buyLimitPriceRatio': Decimal('0.05'), 'sellLimitPriceRatio': Decimal('0.05'), 'feeRateUpRatio': Decimal('0.005'), 'makerFeeRate': Decimal('0.0002'), 'takerFeeRate': Decimal('0.0006'), 'openCostUpRatio': Decimal('0.01'), 'supportMarginCoins': ['USDT'], 'minTradeNum': Decimal('0.0001'), 'maxOrderQty': 1200, 'priceEndStep': 1, 'volumePlace': 4, 'pricePlace': 1, 'sizeMultiplier': Decimal('0.0001'), 'symbolType': 'perpetual', 'minTradeUSDT': Decimal('5'), 'maxSymbolOrderNum': 200, 'maxProductOrderNum': 1000, 'maxPositionNum': 200, 'symbolStatus': 'normal', 'offTime': datetime.

In [10]:
async def perp_open_orders(symbol: str) -> list[OrderState]:
  raw = await client.classic.mix.order.open(product_type=PRODUCT_TYPE, symbol=symbol)
  out: list[OrderState] = []
  for o in raw['entrustedList'] or []:
    size = Decimal(o['size'])
    filled = Decimal(o['baseVolume'])
    sign = 1 if o['side'] == 'buy' else -1
    out.append(OrderState(
      id=o['orderId'],
      price=Decimal(o['price']),
      qty=sign * size,
      filled_qty=sign * filled,
      active=o['status'] in ('live', 'partially_filled'),
      details=o,
    ))
  return out

{symbol: await perp_open_orders(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [4]:
async def perp_trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.classic.mix.order.fills(
    product_type=PRODUCT_TYPE, symbol=symbol, start_time=start, end_time=end,
  )
  out: list[Trade] = []
  for f in raw['fillList'] or []:
    fee_amount = sum((abs(Decimal(d['totalFee'])) for d in f['feeDetail']), Decimal(0))
    fee_asset = f['feeDetail'][0]['feeCoin'] if f['feeDetail'] else None
    size = Decimal(f['baseVolume'])
    out.append(Trade(
      id=f['tradeId'],
      price=Decimal(f['price']),
      qty=size if f['side'] == 'buy' else -size,
      time=f['cTime'],
      maker=f['tradeScope'] == 'maker',
      fee=Trade.Fee(amount=fee_amount, asset=fee_asset) if fee_amount and fee_asset else None,
      details=f,
    ))
  return out

{symbol: await perp_trades_history(symbol, start, end) for symbol in PERP_MARKETS}

{'BTCUSDT': [], 'ETHUSDT': [], 'SOLUSDT': []}

In [5]:
async def index(symbol: str, *, settings: Settings = {}) -> Decimal:
  raw = await client.classic.mix.market.symbol_price(symbol, product_type=PRODUCT_TYPE)
  return raw[0]['indexPrice']

{symbol: await index(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': Decimal('79529.4185'),
 'ETHUSDT': Decimal('2477.428'),
 'SOLUSDT': Decimal('105.3315')}

In [13]:
async def next_funding(symbol: str) -> NextFunding:
  rate, time_info = await asyncio.gather(
    client.classic.mix.market.funding.current_rate(symbol=symbol, product_type=PRODUCT_TYPE),
    client.classic.mix.market.funding.time(symbol=symbol, product_type=PRODUCT_TYPE),
  )
  return NextFunding(
    rate=rate[0]['fundingRate'],
    time=time_info[0]['nextFundingTime'],
    interval=timedelta(hours=int(rate[0]['fundingRateInterval'])),
  )

{symbol: await next_funding(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': NextFunding(rate=Decimal('0.000068'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'ETHUSDT': NextFunding(rate=Decimal('0.000084'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'SOLUSDT': NextFunding(rate=Decimal('0.000075'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800))}

In [14]:
async def funding_rates(
  symbol: str, start: datetime | None = None, end: datetime | None = None,
) -> list[FundingRate]:
  # `rate_history` paginates by page number, not a date range -- filtered client-side.
  raw = await client.classic.mix.market.funding.rate_history(
    symbol=symbol, product_type=PRODUCT_TYPE, page_size=50,
  )
  out = [FundingRate(rate=r['fundingRate'], time=r['fundingTime']) for r in raw]
  if start is not None:
    out = [r for r in out if r.time >= start]
  if end is not None:
    out = [r for r in out if r.time <= end]
  return out

funding_end = datetime.now(timezone.utc)
funding_start = funding_end - timedelta(days=7)
{symbol: await funding_rates(symbol, funding_start, funding_end) for symbol in PERP_MARKETS}

{'BTCUSDT': [FundingRate(rate=Decimal('0.000079'), time=datetime.datetime(2026, 9, 4, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000008'), time=datetime.datetime(2026, 9, 4, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000043'), time=datetime.datetime(2026, 9, 3, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000094'), time=datetime.datetime(2026, 9, 3, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.00004'), time=datetime.datetime(2026, 9, 3, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000069'), time=datetime.datetime(2026, 9, 2, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000067'), time=datetime.datetime(2026, 9, 2, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.0001'), time=datetime.datetime(2026, 9, 2, 0, 0, tzinfo=datetime.time

### `funding_payments` -- not supported

Bitget has no dedicated funding-fee endpoint under `classic.mix`. Both ledgers that would
carry the payments filter on a free-text type field the venue publishes no closed set
for: `classic.mix.account.bills`'s `businessType` and `classic.tax.futures_records`'s
`futureTaxType` (the venue documents that 30+ values exist without listing them). Walking
eight months of this account's `USDT-FUTURES` tax records turned up 15 distinct values
(`buy_deal`, `sell_deal`, `open_long`, `open_short`, `close_short`, `burst_close_long`,
`adjust_margin_ifm`, `contract_margin_settle_fee`, `trans_to_cross`, `trans_to_isolated`,
`trans_to_exchange`, `trans_from_exchange`, `user_grants_issue`, `user_grants_recycle`,
`risk_captital_user_transfer`), none documented as the funding settlement, and this
account has never held a position across a settlement to observe one. Rather than guess a
substring filter, the method raises.

In [ ]:
async def funding_payments(symbol: str, start: datetime, end: datetime) -> list[FundingPayment]:
  raise NotImplementedError(
    'not supported: Bitget publishes no closed set of futures ledger types '
    '(`classic.mix.account.bills` `businessType`, `classic.tax.futures_records` '
    '`futureTaxType`), so there is no documented funding-fee value to filter on'
  )

# not executed: not supported, see the note above
{symbol: await funding_payments(symbol, funding_start, funding_end) for symbol in PERP_MARKETS}

In [16]:
async def perp_position(symbol: str) -> PerpPosition:
  raw = await client.classic.mix.position.get(product_type=PRODUCT_TYPE, symbol=symbol, margin_coin='USDT')
  row = raw[0] if raw else None
  if row is None:
    return PerpPosition()
  size = row['total'] if row['holdSide'] == 'long' else -row['total']
  return PerpPosition(size=size, entry_price=row['openPriceAvg'])

{symbol: await perp_position(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'ETHUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0')),
 'SOLUSDT': PerpPosition(size=Decimal('0'), entry_price=Decimal('0'))}

In [ ]:
async def perp_collateral(symbol: str) -> PerpCollateral:
  raise NotImplementedError(
    'not supported: `MixAccountAsset` carries no initial/maintenance margin figure '
    '(only `available`, `accountEquity` and `crossedRiskRate`), and `PerpCollateral` '
    'requires both'
  )

# not executed: not supported, see the coverage note below
{symbol: await perp_collateral(symbol) for symbol in PERP_MARKETS}

In [6]:
async def perp_account(symbol: str):
  return await client.classic.mix.account.get(
    symbol=symbol, product_type=PRODUCT_TYPE, margin_coin='USDT',
  )


async def perp_available_notional(symbol: str) -> Decimal:
  account, rules = await asyncio.gather(perp_account(symbol), perp_rules(symbol))
  return account['available'] * rules.details['maxLever']

{symbol: await perp_available_notional(symbol) for symbol in PERP_MARKETS}

{'BTCUSDT': Decimal('0.62918550'),
 'ETHUSDT': Decimal('0.62918550'),
 'SOLUSDT': Decimal('0.41945700')}

In [ ]:
async def perp_place_order(symbol: str, order: Order, *, settings: Settings = {}) -> OrderResponse:
  qty = Decimal(str(order['qty']))
  side = 'buy' if qty > 0 else 'sell'
  size = abs(qty)
  price = Decimal(str(order['price']))
  body: MixLimitOrderRequest | MixMarketOrderRequest
  if order['type'] == 'MARKET':
    body = {
      'symbol': symbol, 'productType': PRODUCT_TYPE, 'marginMode': 'crossed', 'marginCoin': 'USDT',
      'side': side, 'orderType': 'market', 'size': size,
    }
  else:
    force = 'post_only' if order['type'] == 'POST_ONLY' else 'gtc'
    body = {
      'symbol': symbol, 'productType': PRODUCT_TYPE, 'marginMode': 'crossed', 'marginCoin': 'USDT',
      'side': side, 'orderType': 'limit', 'force': force, 'price': price, 'size': size,
    }
  raw = await client.classic.mix.order.place(body)
  return OrderResponse(id=raw['orderId'], details=raw)

# Not executed here -- would place a real order on the account.
await perp_place_order('BTCUSDT', {'qty': Decimal('0.001'), 'price': Decimal('20000'), 'type': 'LIMIT'})

In [ ]:
async def perp_cancel_order(symbol: str, id: str, *, settings: Settings = {}):
  return await client.classic.mix.order.cancel(symbol=symbol, product_type=PRODUCT_TYPE, order_id=id)

# Not executed here -- would cancel a real order on the account.
await perp_cancel_order('BTCUSDT', '123456')

### Coverage assessment: `PerpMarket` (Classic Mix)

**Mostly supported.** `depth`, `rules`, `open_orders`, `trades_history`, `index`,
`next_funding`, `funding_rates`, `perp_position` and `available_notional` map onto
dedicated endpoints and are confirmed against live (mostly empty) responses: `index()`
reads `classic.mix.market.symbol_price`'s `indexPrice`, and `available_notional` is the
account's `available` margin times the contract's `maxLever`.

Two methods are not supported on Classic. `perp_collateral` has nothing to fill
`PerpCollateral.initial_margin`/`maintenance_margin` from: `MixAccountAsset` exposes
`available`, `accountEquity` and `crossedRiskRate` but no account-level margin
requirement (UTA's `account.assets` does, see `market/uta.ipynb`). `funding_payments`
has no documented ledger type to filter on (see its note above).

`mix.position.list` returns no open positions for any product type and the USDT-FUTURES
equity is 0.0042 USDT, so the account-scoped mappings are demonstrated, not numerically
exercised. `classic.mix.order.fills` and `classic.mix.account.get` validate on this
one-way, mixed-margin-mode account (`tradeSide` admits `buy_single`/`sell_single`; the
`*UnrealizedPL` fields admit `''`). `classic.mix.order.open`'s `tradeSide` still uses the
narrow `MixTradeSide` alias, so `open_orders` validates only because there are no open
orders -- tracked in `typed-client-issues.md`.